# Regularization

Three models fit a dataset: a line (loss 10), a quadratic (loss 2), and a wild degree-10 polynomial (loss 0.1). By raw loss the wiggly polynomial wins — but something tells us the quadratic is the *real* winner: the degree-10 curve fits the points without getting the point. **Regularization** encodes that instinct: add a **penalty for complexity** — the L2 penalty, the sum of squared coefficients — to the loss, and re-rank. Now the totals read 26, 22, 246.1, and the quadratic wins. It is based on the fifth video of Lesson 6 (Point Estimation), setting up the punchline that regularization *is* Bayesian probability in disguise. 

## 1. Introduction

A model that fits the training data best is not necessarily the best model. The degree-10 polynomial threads every point, yet it is chaotic — it "doesn't really get the point of the data, which is kind of a quadratic that points upwards." Trusting it would be **overfitting**: excellent memory, poor generalization.

Machine learning's everyday cure is to charge models for their complexity. This notebook builds that cure — **L2 regularization** — with the video's exact numbers, and leaves a deliberate cliffhanger: *what does this have to do with probability and maximum likelihood?* (Answer in the next video: everything.)

After completing this notebook you will be able to:

- Diagnose overfitting from the fit-vs-generalization gap.
- Compute the **L2 penalty** (sum of squared coefficients, constant excluded) and the **regularized loss** $\text{loss} + \lambda \cdot \text{penalty}$.
- Re-rank the three models the video's way, and study how the **regularization parameter** $\lambda$ arbitrates fit vs. simplicity.
- Verify with held-out data that the regularized choice generalizes better.

## 2. Theory

### Three models, one dataset

Candidates for the same points:

1. **Model 1 (linear):** $y = 4x + 3$ — squared-error loss $10$ (doesn't fit that well),
2. **Model 2 (quadratic):** $y = 2x^2 - 4x + 5$ — loss $2$ (fits nicely),
3. **Model 3 (degree 10):** the wiggly $y = 4x^{10} - 9x^8 - 2x^6 + 3x^5 - 6x^4 - 10x + 4$ — loss $0.1$ (fits *really* well).

Raw loss crowns model 3. But the picture says otherwise: model 3 is chaotic, contorting itself through every point. We want a mechanism that lets model 2 win.

### The L2 penalty

Charge each model for its coefficients: the **L2 penalty** is the sum of the squares of all polynomial coefficients — *except the constant term*:

$$
\text{model 1: } 4^2 = 16 \qquad
\text{model 2: } 2^2 + (-4)^2 = 20 \qquad
\text{model 3: } 4^2 + (-9)^2 + (-2)^2 + 3^2 + (-6)^2 + (-10)^2 = 246
$$

More complicated models — many large coefficients — pay more.

### The regularized loss

$$
\text{new loss} = \text{old loss} + \text{penalty}
$$

| model | loss | L2 penalty | total |
|---|---|---|---|
| linear | 10 | 16 | 26 |
| quadratic | 2 | 20 | **22** |
| degree 10 | 0.1 | 246 | 246.1 |

**Model 2 wins.** We modified the loss to penalize models that are too complex — seeking the *simplest model that fits the data well*.

### The regularization parameter

Sometimes the full penalty is too drastic. Scale it by a small number $\lambda$, the **regularization parameter**:

$$
\text{regularized error} = \text{LL} + \lambda \cdot \sum_{i \ge 1} a_i^2
$$

for a polynomial $y = a_n x^n + \dots + a_1 x + a_0$ with loss LL (the video's "log loss" — any loss works). $\lambda = 0$ recovers pure loss-minimization; large $\lambda$ crushes all coefficients toward zero; in between, $\lambda$ dials the fit-simplicity trade-off.

### The cliffhanger

That is regularization in a nutshell. But what does adding a sum of squared coefficients to a loss have to do with **probability and maximum likelihood**? The next video's beautiful answer: the penalty is a *prior* — regularized estimation is **maximum a posteriori** estimation.

## 3. Worked Examples

### Example 1: the penalties, line by line

Model 1 ($4x + 3$): only the slope counts → $16$. Model 2 ($2x^2 - 4x + 5$): $4 + 16 = 20$. Model 3: $16 + 81 + 4 + 9 + 36 + 100 = 246$. Note the constant terms ($3$, $5$, …) are never charged — shifting a curve up or down changes nothing about its wiggliness.

### Example 2: what λ does to the ranking

With totals $\text{loss} + \lambda \cdot \text{penalty}$:

- $\lambda = 0$: ranking by pure loss → model 3 wins (overfit).
- $\lambda = 1$ (the video's case): 26 / 22 / 246.1 → model 2.
- $\lambda$ huge: penalties dominate → model 1 (16 is the smallest penalty), and eventually the constant model.

Somewhere around $\lambda \approx 0.008$, models 2 and 3 swap places: $2 + 20\lambda = 0.1 + 246\lambda \Rightarrow \lambda = \tfrac{1.9}{226} \approx 0.0084$. Even a *tiny* complexity charge dethrones the wiggle.

### Example 3: why the quadratic "gets the point"

On *fresh* data from the same source, the degree-10 polynomial's wiggles land nowhere near the new points — its test loss explodes while the quadratic's stays near its training loss. Generalization, not training fit, is what the penalty is a proxy for.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

# a dataset whose 'point' is a quadratic pointing upwards
n_pts = 11
x = np.linspace(-2, 3, n_pts)
y = 2 * x**2 - 4 * x + 5 + rng.normal(0, 1.6, n_pts)

### Fit the three models and score them

In [ ]:
fits = {name: np.polyfit(x, y, deg)
        for name, deg in [("model 1 (linear)", 1),
                          ("model 2 (quadratic)", 2),
                          ("model 3 (degree 10)", 10)]}

def sq_loss(coeffs):
    return np.mean((y - np.polyval(coeffs, x))**2)

def l2_penalty(coeffs):
    return np.sum(coeffs[:-1]**2)          # all coefficients EXCEPT the constant

print(f"{'model':<22} {'loss':>8} {'penalty':>9} {'loss+penalty':>13}")
for name, c in fits.items():
    print(f"{name:<22} {sq_loss(c):>8.2f} {l2_penalty(c):>9.1f} "
          f"{sq_loss(c) + l2_penalty(c):>13.1f}")
print("\nRaw loss crowns model 3; the regularized total crowns model 2 --")
print("the same reversal as the video's 10/2/0.1 -> 26/22/246.1 table.")

### The video's exact numbers

In [ ]:
video = {"model 1 (linear)": (10.0, 16.0),
         "model 2 (quadratic)": (2.0, 20.0),
         "model 3 (degree 10)": (0.1, 246.0)}

print(f"{'model':<22} {'loss':>6} {'penalty':>8} {'total':>7}")
for name, (loss, pen) in video.items():
    print(f"{name:<22} {loss:>6} {pen:>8} {loss + pen:>7.1f}")
winner = min(video, key=lambda m: sum(video[m]))
print(f"\nwinner: {winner}")

### The generalization check: train loss vs test loss

Fresh points from the same quadratic-plus-noise source expose what the penalty was standing in for.

In [ ]:
rng = np.random.default_rng(7)

x_test = rng.uniform(-2, 3, 200)
y_test = 2 * x_test**2 - 4 * x_test + 5 + rng.normal(0, 1.6, 200)

print(f"{'model':<22} {'train loss':>11} {'test loss':>11}")
for name, c in fits.items():
    test = np.mean((y_test - np.polyval(c, x_test))**2)
    print(f"{name:<22} {sq_loss(c):>11.2f} {test:>11.2f}")
print("\nModel 3 wins training and loses testing by a mile: overfitting made")
print("measurable. The complexity penalty predicted this without any test data.")

### Sweeping the regularization parameter

In [ ]:
lambdas = np.logspace(-4, 2, 400)
names = list(video)
totals = np.array([[video[m][0] + lam * video[m][1] for m in names]
                   for lam in lambdas])
winners = np.array(names)[np.argmin(totals, axis=1)]

changes = [0] + [i for i in range(1, len(lambdas)) if winners[i] != winners[i-1]]
print("winner as lambda grows:")
for i in changes:
    print(f"  lambda >= {lambdas[i]:.4f}: {winners[i]}")
print(f"\ncrossover 3 -> 2 at lambda = 1.9/226 = {1.9/226:.4f}")

## 5. Visualizations

### The classic picture: underfit, good fit, overfit

In [ ]:
grid = np.linspace(-2.15, 3.15, 500)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
for ax, (name, c) in zip(axes, fits.items()):
    ax.scatter(x, y, s=45, color="black", zorder=3)
    ax.plot(grid, np.polyval(c, grid), color="tab:red", linewidth=2)
    ax.set_title(f"{name}\nloss {sq_loss(c):.2f}, penalty {l2_penalty(c):.0f}")
    ax.set_xlabel("x")
    ax.set_ylim(y.min() - 4, y.max() + 4)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("y")
plt.suptitle("Fitting the data vs getting the point: the degree-10 wiggle is chaos between the dots")
plt.tight_layout()
plt.show()

### Loss, penalty, total — the video's table as bars

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
idx = np.arange(3)
losses = [video[m][0] for m in names]
pens = [video[m][1] for m in names]

ax.bar(idx, losses, 0.6, label="loss", color="tab:blue")
ax.bar(idx, pens, 0.6, bottom=losses, label="L2 penalty", color="tab:orange")
for i, m in enumerate(names):
    total = losses[i] + pens[i]
    ax.annotate(f"{total:.1f}", xy=(i, total), xytext=(i, total + 6),
                ha="center", fontweight="bold")
ax.set_xticks(idx)
ax.set_xticklabels(["linear", "quadratic", "degree 10"])
ax.set_ylabel("loss + penalty")
ax.set_title("Regularized totals: 26 vs 22 vs 246.1 -- the quadratic wins")
ax.grid(True, alpha=0.3, axis="y")
ax.legend()
plt.show()

### The λ dial: every model has its regime

In [ ]:
plt.figure(figsize=(9.5, 5))
for m, color in zip(names, ["tab:blue", "tab:green", "tab:red"]):
    plt.plot(lambdas, video[m][0] + lambdas * video[m][1], color=color,
             linewidth=2, label=m)
plt.axvline(1.9 / 226, color="gray", linestyle=":", label="crossover 3 -> 2")
plt.axvline(1.0, color="black", linestyle="--", linewidth=1,
            label="the video's lambda = 1")
plt.xscale("log")
plt.yscale("log")
plt.xlabel("regularization parameter $\\lambda$ (log)")
plt.ylabel("regularized error (log)")
plt.title("loss + $\\lambda$ * penalty: tiny $\\lambda$ favors the overfitter, large $\\lambda$ the simplest")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.show()

### The whole story on one page

Everything above in a single sheet: the three models, the loss-plus-penalty table, the λ dial, and the train-versus-test check.

<img src="img/05_regularization.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

**Exercise 1.** Compute the L2 penalty of $y = 3x^3 - 2x^2 + 5x - 7$, and the regularized error at $\lambda = 1$ and $\lambda = 0.1$ if its loss is 1.5.

**Exercise 2.** Why is the constant coefficient excluded from the penalty? (Hint: what does a model with huge $a_0$ but tiny other coefficients look like — is it complex?)

### Intermediate

**Exercise 3.** Find, analytically and by code, the exact $\lambda$ ranges in which each of the video's three models wins. Which model can *never* win, no matter the $\lambda$? (Careful — check all three pairwise crossovers.)

**Exercise 4.** Ridge in action: refit the degree-10 polynomial with an L2 penalty inside the fitting itself (solve the regularized least-squares normal equations $(X^TX + \lambda I)w = X^Ty$ on the Vandermonde matrix, leaving the intercept unpenalized). Plot the fitted curve for $\lambda = 0, 0.1, 10$ — what happens to the wiggles?

### Advanced

**Exercise 5.** L1 vs L2: recompute the penalties of the three video models using the **L1 penalty** (sum of *absolute* coefficients). Does the ranking change at $\lambda = 1$? Then, on the ridge fit of Exercise 4, compare how L2 and a crude L1 (soft-threshold) treatment affect *small* coefficients — which one produces exact zeros, and why is that prized in practice?

**Exercise 6.** Choose $\lambda$ honestly: split the data (train on 8 points, validate on 3), fit ridge polynomials of degree 10 for a grid of $\lambda$, and pick the $\lambda$ minimizing *validation* loss. Compare the winning curve with the true quadratic. What everyday ML procedure did you just perform?

## 7. Solutions

In [ ]:
# Exercise 1: penalty of a cubic
coeffs = np.array([3, -2, 5, -7])          # highest degree first
pen = np.sum(coeffs[:-1]**2)
print(f"Exercise 1: penalty = 3^2 + (-2)^2 + 5^2 = {pen}")
for lam in [1, 0.1]:
    print(f"  lambda = {lam}: regularized error = 1.5 + {lam} * {pen} = "
          f"{1.5 + lam * pen}")

In [ ]:
# Exercise 2: why spare the constant
print("Exercise 2: the constant a_0 only shifts the curve up or down -- it adds")
print("no wiggle, no slope, no complexity. A model y = 1000 is as 'simple' as")
print("y = 0. Penalizing a_0 would also make the fit depend on where the y-axis")
print("zero happens to be (e.g. temperatures in C vs K), which no sensible")
print("complexity measure should do.")

In [ ]:
# Exercise 3: winning ranges of lambda
# totals: m1 = 10 + 16 L, m2 = 2 + 20 L, m3 = 0.1 + 246 L
# m3 vs m2: L < 1.9/226 = 0.0084 -> m3 wins
# m2 vs m1: 2 + 20 L < 10 + 16 L  <=>  4 L < 8  <=>  L < 2 -> m2 beats m1
# m1 wins for L > 2 (and beats m3 whenever L > 9.9/230 = 0.043)
for lam in [0.001, 0.008, 0.01, 1, 2, 3, 100]:
    totals_ = {m: video[m][0] + lam * video[m][1] for m in video}
    print(f"  lambda = {lam:>7}: winner = {min(totals_, key=totals_.get)}")
print("\nExercise 3: model 3 wins for lambda < 0.0084; model 2 for")
print("0.0084 < lambda < 2; model 1 for lambda > 2. Every model has a regime")
print("here -- though with these numbers the LINE never beats the quadratic")
print("until lambda = 2, where simplicity finally outweighs its poor fit.")

In [ ]:
# Exercise 4: ridge-regularized degree-10 fit
def ridge_polyfit(x, y, deg, lam):
    X = np.vander(x, deg + 1)               # columns x^deg ... x^1, 1
    penalty_mask = np.ones(deg + 1)
    penalty_mask[-1] = 0                     # leave the intercept unpenalized
    A = X.T @ X + lam * np.diag(penalty_mask)
    return np.linalg.solve(A, X.T @ y)

grid = np.linspace(-2.15, 3.15, 500)
plt.figure(figsize=(9.5, 5))
plt.scatter(x, y, s=45, color="black", zorder=3, label="data")
for lam, color in [(0, "tab:red"), (0.1, "tab:orange"), (10, "tab:green")]:
    w = ridge_polyfit(x, y, 10, lam)
    plt.plot(grid, np.polyval(w, grid), color=color, linewidth=2,
             label=f"degree 10, lambda = {lam}")
plt.ylim(y.min() - 4, y.max() + 4)
plt.title("Exercise 4: the penalty inside the fit irons out the wiggles")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print("lambda = 0 reproduces the chaotic interpolator; growing lambda shrinks")
print("the high-order coefficients and the degree-10 model relaxes toward the")
print("smooth quadratic shape the data actually follows.")

In [ ]:
# Exercise 5: L1 vs L2
print("Exercise 5: L1 penalties (video's coefficient sets, constants excluded):")
l1 = {"model 1": abs(4), "model 2": abs(2) + abs(-4)}
print(f"  model 1: |4| = {l1['model 1']}   model 2: |2|+|-4| = {l1['model 2']}")
print("  model 3: |4|+|-9|+|-2|+|3|+|-6|+|-10| = 34 -- still the biggest,")
print("  so at lambda = 1 the ranking is unchanged: quadratic wins (2+6 < 0.1+34).")

w0 = ridge_polyfit(x, y, 10, 0.1)
soft = np.sign(w0) * np.maximum(np.abs(w0) - 0.05, 0)   # crude L1 shrinkage
print(f"\nridge (L2) small coefficients:   {np.round(np.sort(np.abs(w0))[:4], 4)}")
print(f"after L1 soft-threshold:         {np.round(np.sort(np.abs(soft))[:4], 4)}")
print("L2 shrinks small coefficients but never to exactly zero; L1 subtracts a")
print("constant and clips at zero, producing EXACT zeros -- automatic feature")
print("selection, the reason lasso is prized for sparse models.")

In [ ]:
# Exercise 6: choosing lambda by validation
rng = np.random.default_rng(3)

perm = rng.permutation(n_pts)
tr, va = perm[:8], perm[8:]

lams = np.logspace(-4, 2, 100)
val_losses = []
for lam in lams:
    w = ridge_polyfit(x[tr], y[tr], 10, lam)
    val_losses.append(np.mean((y[va] - np.polyval(w, x[va]))**2))
best_lam = lams[int(np.argmin(val_losses))]
print(f"Exercise 6: best lambda by validation = {best_lam:.3f} "
      f"(val loss {min(val_losses):.2f})")

w_best = ridge_polyfit(x[tr], y[tr], 10, best_lam)
plt.figure(figsize=(9, 4.5))
plt.scatter(x[tr], y[tr], s=45, color="black", label="train")
plt.scatter(x[va], y[va], s=60, color="tab:red", marker="s", label="validation")
plt.plot(grid, np.polyval(w_best, grid), color="tab:green", linewidth=2,
         label=f"degree 10, lambda = {best_lam:.3f}")
plt.plot(grid, 2 * grid**2 - 4 * grid + 5, "--", color="gray", label="true quadratic")
plt.ylim(y.min() - 4, y.max() + 4)
plt.title("Validation-chosen ridge fit hugs the true curve")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print("This is hyperparameter tuning by hold-out validation -- the standard ML")
print("recipe for setting regularization strength.")

## 8. Conclusion

- Best training fit ≠ best model: the degree-10 polynomial's loss of 0.1 beat the quadratic's 2, yet it is chaotic — fitting the data without getting its point. That gap is **overfitting**.
- **L2 regularization** charges for complexity: penalty = sum of squared coefficients (constant excluded — shifts aren't complexity). The video's numbers: 16, 20, 246.
- **Regularized error = loss + λ·penalty** re-ranks the field: 26 / **22** / 246.1 → the quadratic wins; the parameter $\lambda$ dials between pure fit ($\lambda = 0$) and pure simplicity (large $\lambda$), and even a tiny $\lambda$ (≈ 0.008) dethrones the overfitter.
- Held-out data confirms what the penalty predicted: the wiggly model's test loss explodes while the quadratic generalizes — and validation is the honest way to choose $\lambda$ (with L1 as the sparsity-producing sibling of L2).
- Cliffhanger, as posed by the video: what does adding squared coefficients to a loss have to do with probability and maximum likelihood? Next notebook: the penalty is a **prior**, and regularized estimation is **MAP**.